# Cohort Generation with OMOPy

This notebook covers **cohort generation** using `omopy.connector` — the process
of defining study populations from an OMOP CDM.

Topics covered:
1. **Concept-based cohorts** — `generate_concept_cohort_set` with codelists
2. **Cohort metadata** — settings, attrition, codelist, counts
3. **Multiple cohorts** — generating several cohorts in one call
4. **CIRCE/JSON cohort definitions** — `generate_cohort_set` with ATLAS-style JSON
5. **Subsetting CDM by cohort** — `cdm_subset_cohort`

## Configuration

Set the path to your DuckDB file and schema names here. All subsequent cells
reference these variables.

In [1]:
# -- Configuration --------------------------------------------------------
DB_PATH = "../data/synthea.duckdb"  # path to DuckDB file (relative to this notebook)
CDM_SCHEMA = "base"  # schema containing the OMOP CDM tables
WRITE_SCHEMA = "main"  # schema for writing cohort tables
# -------------------------------------------------------------------------

In [2]:
from omopy.connector import (
    cdm_from_con,
    cdm_subset_cohort,
    generate_cohort_set,
    generate_concept_cohort_set,
)
from omopy.generics import Codelist

## Connect to CDM

Cohort generation writes temporary and result tables, so we provide a
`write_schema` in addition to the read-only `cdm_schema`.

In [3]:
cdm = cdm_from_con(
    DB_PATH,
    cdm_schema=CDM_SCHEMA,
    write_schema=WRITE_SCHEMA,
)
print(cdm)
print(f"Persons: {cdm['person'].count()}")
print(f"Cohort tables: {cdm.cohort_tables}")

CdmReference(name='dbt-synthea', version=5.4, source=duckdb, tables=36)
Persons: 27
Cohort tables: {}


## Creating Codelists for Cohorts

A `Codelist` maps concept-set names to lists of OMOP concept IDs. Each name
becomes one cohort definition when passed to `generate_concept_cohort_set`.

We will use two conditions present in the Synthea database:
- **Viral sinusitis** — concept 40481087 (4 occurrences)
- **Essential hypertension** — concept 320128 (6 occurrences)

In [4]:
# Single-concept codelist
sinusitis_codes = Codelist({"viral_sinusitis": [40481087]})
print(sinusitis_codes)
print("Names:", sinusitis_codes.names)
print("Concept IDs:", sinusitis_codes.all_concept_ids)

Codelist(1 codelist(s), 1 total concept ID(s))
Names: ['viral_sinusitis']
Concept IDs: {40481087}


In [5]:
# A plain dict also works — it is converted to Codelist internally
hypertension_codes = {"essential_hypertension": [320128]}
print(hypertension_codes)

{'essential_hypertension': [320128]}


## Concept-Based Cohort Generation

`generate_concept_cohort_set` builds cohorts from concept sets by:
1. Looking up the concept's domain (Condition, Drug, etc.)
2. Finding matching clinical events in the appropriate table
3. Constraining to observation periods
4. Applying limit (first event or all events per person)
5. Collapsing overlapping periods

```python
generate_concept_cohort_set(
    cdm,
    concept_set,           # Codelist or dict
    name,                  # name for the cohort table
    *,
    limit="first",         # "first" or "all"
    end="observation_period_end_date",  # or "event_end_date" or int (days)
    required_observation=(0, 0),  # (prior_days, future_days)
)
```

### Basic cohort: first event, end at observation period

In [6]:
# Generate a cohort for hypertension using default settings
cdm = generate_concept_cohort_set(
    cdm,
    hypertension_codes,
    "hypertension",
)

# The function returns the CDM with the new cohort table attached
print("Cohort tables:", cdm.cohort_tables)
print(cdm["hypertension"])

Cohort tables: {'hypertension': CohortTable('hypertension', source='duckdb', cohorts=1)}
CohortTable('hypertension', source='duckdb', cohorts=1)


In [7]:
# Collect the cohort data as a Polars DataFrame
ht_df = cdm["hypertension"].collect()
print(f"Records: {len(ht_df)}")
print(f"Columns: {ht_df.columns}")
print(ht_df)

Records: 6
Columns: ['cohort_definition_id', 'subject_id', 'cohort_start_date', 'cohort_end_date']
shape: (6, 4)
┌──────────────────────┬────────────┬───────────────────┬─────────────────┐
│ cohort_definition_id ┆ subject_id ┆ cohort_start_date ┆ cohort_end_date │
│ ---                  ┆ ---        ┆ ---               ┆ ---             │
│ i64                  ┆ i64        ┆ date              ┆ date            │
╞══════════════════════╪════════════╪═══════════════════╪═════════════════╡
│ 1                    ┆ 16         ┆ 1989-07-09        ┆ 2024-01-21      │
│ 1                    ┆ 19         ┆ 2003-09-08        ┆ 2024-01-01      │
│ 1                    ┆ 17         ┆ 2000-08-12        ┆ 2023-12-24      │
│ 1                    ┆ 5          ┆ 1991-12-22        ┆ 2023-10-29      │
│ 1                    ┆ 15         ┆ 2005-06-29        ┆ 2024-01-24      │
│ 1                    ┆ 21         ┆ 1973-08-15        ┆ 2023-05-24      │
└──────────────────────┴────────────┴──────────────

### Cohort with `limit="all"` — keep all events

In [8]:
# Keep all qualifying events per person, not just the first
cdm = generate_concept_cohort_set(
    cdm,
    sinusitis_codes,
    "sinusitis_all",
    limit="all",
)

sinus_df = cdm["sinusitis_all"].collect()
print(f"Records with limit='all': {len(sinus_df)}")
print(sinus_df)

Records with limit='all': 3
shape: (3, 4)
┌──────────────────────┬────────────┬───────────────────┬─────────────────┐
│ cohort_definition_id ┆ subject_id ┆ cohort_start_date ┆ cohort_end_date │
│ ---                  ┆ ---        ┆ ---               ┆ ---             │
│ i64                  ┆ i64        ┆ date              ┆ date            │
╞══════════════════════╪════════════╪═══════════════════╪═════════════════╡
│ 1                    ┆ 2          ┆ 2023-06-17        ┆ 2023-10-26      │
│ 1                    ┆ 17         ┆ 2023-10-02        ┆ 2023-12-24      │
│ 1                    ┆ 16         ┆ 2023-04-28        ┆ 2024-01-21      │
└──────────────────────┴────────────┴───────────────────┴─────────────────┘


### Cohort with `end="event_end_date"` — use the clinical event's end date

In [9]:
# End the cohort at the clinical event's end date instead of observation period end
cdm = generate_concept_cohort_set(
    cdm,
    hypertension_codes,
    "ht_event_end",
    end="event_end_date",
)

event_end_df = cdm["ht_event_end"].collect()
print("End = event_end_date:")
print(event_end_df)

End = event_end_date:
shape: (6, 4)
┌──────────────────────┬────────────┬───────────────────┬─────────────────┐
│ cohort_definition_id ┆ subject_id ┆ cohort_start_date ┆ cohort_end_date │
│ ---                  ┆ ---        ┆ ---               ┆ ---             │
│ i64                  ┆ i64        ┆ date              ┆ date            │
╞══════════════════════╪════════════╪═══════════════════╪═════════════════╡
│ 1                    ┆ 17         ┆ 2000-08-12        ┆ 2000-08-12      │
│ 1                    ┆ 19         ┆ 2003-09-08        ┆ 2003-09-08      │
│ 1                    ┆ 16         ┆ 1989-07-09        ┆ 1989-07-09      │
│ 1                    ┆ 5          ┆ 1991-12-22        ┆ 1991-12-22      │
│ 1                    ┆ 21         ┆ 1973-08-15        ┆ 1973-08-15      │
│ 1                    ┆ 15         ┆ 2005-06-29        ┆ 2005-06-29      │
└──────────────────────┴────────────┴───────────────────┴─────────────────┘


### Cohort with fixed-duration end — `end=30` (30 days after index)

In [10]:
# End the cohort 30 days after the index date
cdm = generate_concept_cohort_set(
    cdm,
    hypertension_codes,
    "ht_30d",
    end=30,
)

day30_df = cdm["ht_30d"].collect()
print("End = 30 days after index:")
print(day30_df)

End = 30 days after index:
shape: (6, 4)
┌──────────────────────┬────────────┬───────────────────┬─────────────────┐
│ cohort_definition_id ┆ subject_id ┆ cohort_start_date ┆ cohort_end_date │
│ ---                  ┆ ---        ┆ ---               ┆ ---             │
│ i64                  ┆ i64        ┆ date              ┆ date            │
╞══════════════════════╪════════════╪═══════════════════╪═════════════════╡
│ 1                    ┆ 19         ┆ 2003-09-08        ┆ 2003-10-08      │
│ 1                    ┆ 17         ┆ 2000-08-12        ┆ 2000-09-11      │
│ 1                    ┆ 21         ┆ 1973-08-15        ┆ 1973-09-14      │
│ 1                    ┆ 15         ┆ 2005-06-29        ┆ 2005-07-29      │
│ 1                    ┆ 16         ┆ 1989-07-09        ┆ 1989-08-08      │
│ 1                    ┆ 5          ┆ 1991-12-22        ┆ 1992-01-21      │
└──────────────────────┴────────────┴───────────────────┴─────────────────┘


### Required observation — minimum time before/after index

The `required_observation` parameter filters out persons who do not have
sufficient observation time around the index date. It is a tuple of
`(prior_days, future_days)`.

In [11]:
# Require at least 365 days of prior observation
cdm = generate_concept_cohort_set(
    cdm,
    hypertension_codes,
    "ht_365",
    required_observation=(365, 0),
)

full_df = cdm["hypertension"].collect()
restricted_df = cdm["ht_365"].collect()
print(f"Without observation requirement: {len(full_df)} records")
print(f"With 365-day prior observation:  {len(restricted_df)} records")

Without observation requirement: 6 records
With 365-day prior observation:  4 records


## Exploring Cohort Results

A `CohortTable` (returned by `cdm["cohort_name"]`) inherits from `CdmTable`
and adds cohort-specific metadata:

| Property | Description |
|---|---|
| `.collect()` | Materialise as Polars DataFrame |
| `.count()` | Number of rows |
| `.cohort_count()` | Records and subjects per cohort_definition_id |
| `.settings` | Cohort settings (definition_id, name, parameters) |
| `.attrition` | Step-by-step inclusion/exclusion counts |
| `.cohort_codelist` | Concept IDs used per cohort definition |
| `.cohort_ids` | List of cohort_definition_id values |
| `.cohort_names` | List of cohort name strings |

In [12]:
cohort = cdm["hypertension"]
print(cohort)
print(f"Row count: {cohort.count()}")

CohortTable('hypertension', source='duckdb', cohorts=1)
Row count: 6


### Cohort settings

In [13]:
# Settings map cohort_definition_id to cohort_name and generation parameters
print(cohort.settings)

shape: (1, 6)
┌──────────────────┬─────────────────┬───────┬─────────────────┬─────────────────┬─────────────────┐
│ cohort_definitio ┆ cohort_name     ┆ limit ┆ prior_observati ┆ future_observat ┆ end             │
│ n_id             ┆ ---             ┆ ---   ┆ on              ┆ ion             ┆ ---             │
│ ---              ┆ str             ┆ str   ┆ ---             ┆ ---             ┆ str             │
│ i64              ┆                 ┆       ┆ i64             ┆ i64             ┆                 │
╞══════════════════╪═════════════════╪═══════╪═════════════════╪═════════════════╪═════════════════╡
│ 1                ┆ essential_hyper ┆ first ┆ 0               ┆ 0               ┆ observation_per │
│                  ┆ tension         ┆       ┆                 ┆                 ┆ iod_end_date    │
└──────────────────┴─────────────────┴───────┴─────────────────┴─────────────────┴─────────────────┘


### Cohort attrition

In [14]:
# Attrition tracks how many records/subjects were included or excluded at each step
print(cohort.attrition)

shape: (1, 7)
┌──────────────┬──────────────┬──────────────┬───────────┬─────────────┬─────────────┬─────────────┐
│ cohort_defin ┆ number_recor ┆ number_subje ┆ reason_id ┆ reason      ┆ excluded_re ┆ excluded_su │
│ ition_id     ┆ ds           ┆ cts          ┆ ---       ┆ ---         ┆ cords       ┆ bjects      │
│ ---          ┆ ---          ┆ ---          ┆ i64       ┆ str         ┆ ---         ┆ ---         │
│ i64          ┆ i64          ┆ i64          ┆           ┆             ┆ i64         ┆ i64         │
╞══════════════╪══════════════╪══════════════╪═══════════╪═════════════╪═════════════╪═════════════╡
│ 1            ┆ 6            ┆ 6            ┆ 1         ┆ Initial     ┆ 0           ┆ 0           │
│              ┆              ┆              ┆           ┆ qualifying  ┆             ┆             │
│              ┆              ┆              ┆           ┆ events      ┆             ┆             │
└──────────────┴──────────────┴──────────────┴───────────┴─────────────┴─────

### Cohort codelist

In [15]:
# The codelist records which concept IDs were used for each cohort definition
print(cohort.cohort_codelist)

shape: (1, 4)
┌──────────────────────┬────────────────────────┬────────────┬───────────────┐
│ cohort_definition_id ┆ codelist_name          ┆ concept_id ┆ codelist_type │
│ ---                  ┆ ---                    ┆ ---        ┆ ---           │
│ i64                  ┆ str                    ┆ i64        ┆ str           │
╞══════════════════════╪════════════════════════╪════════════╪═══════════════╡
│ 1                    ┆ essential_hypertension ┆ 320128     ┆ index event   │
└──────────────────────┴────────────────────────┴────────────┴───────────────┘


### Cohort count

In [16]:
# Summarise records and unique subjects per cohort definition
print(cohort.cohort_count())

shape: (1, 3)
┌──────────────────────┬────────────────┬─────────────────┐
│ cohort_definition_id ┆ number_records ┆ number_subjects │
│ ---                  ┆ ---            ┆ ---             │
│ i64                  ┆ u32            ┆ u32             │
╞══════════════════════╪════════════════╪═════════════════╡
│ 1                    ┆ 6              ┆ 6               │
└──────────────────────┴────────────────┴─────────────────┘


In [17]:
# Cohort IDs and names from settings
print("Cohort IDs:  ", cohort.cohort_ids)
print("Cohort names:", cohort.cohort_names)

Cohort IDs:   [1]
Cohort names: ['essential_hypertension']


## Multiple Cohorts

A `Codelist` with multiple entries generates one cohort per entry, all stored
in the same `CohortTable`. Each entry gets a unique `cohort_definition_id`.

In [18]:
# Two conditions in one codelist
multi_codes = Codelist(
    {
        "viral_sinusitis": [40481087],
        "essential_hypertension": [320128],
    }
)

cdm = generate_concept_cohort_set(cdm, multi_codes, "multi_cohort")

multi = cdm["multi_cohort"]
print(multi)

CohortTable('multi_cohort', source='duckdb', cohorts=2)


In [19]:
# Settings show both cohort definitions
print(multi.settings)

shape: (2, 6)
┌──────────────────┬─────────────────┬───────┬─────────────────┬─────────────────┬─────────────────┐
│ cohort_definitio ┆ cohort_name     ┆ limit ┆ prior_observati ┆ future_observat ┆ end             │
│ n_id             ┆ ---             ┆ ---   ┆ on              ┆ ion             ┆ ---             │
│ ---              ┆ str             ┆ str   ┆ ---             ┆ ---             ┆ str             │
│ i64              ┆                 ┆       ┆ i64             ┆ i64             ┆                 │
╞══════════════════╪═════════════════╪═══════╪═════════════════╪═════════════════╪═════════════════╡
│ 1                ┆ viral_sinusitis ┆ first ┆ 0               ┆ 0               ┆ observation_per │
│                  ┆                 ┆       ┆                 ┆                 ┆ iod_end_date    │
│ 2                ┆ essential_hyper ┆ first ┆ 0               ┆ 0               ┆ observation_per │
│                  ┆ tension         ┆       ┆                 ┆             

In [20]:
# Count per cohort definition
print(multi.cohort_count())

shape: (2, 3)
┌──────────────────────┬────────────────┬─────────────────┐
│ cohort_definition_id ┆ number_records ┆ number_subjects │
│ ---                  ┆ ---            ┆ ---             │
│ i64                  ┆ u32            ┆ u32             │
╞══════════════════════╪════════════════╪═════════════════╡
│ 1                    ┆ 3              ┆ 3               │
│ 2                    ┆ 6              ┆ 6               │
└──────────────────────┴────────────────┴─────────────────┘


In [21]:
# Attrition is tracked separately per definition
print(multi.attrition)

shape: (2, 7)
┌──────────────┬──────────────┬──────────────┬───────────┬─────────────┬─────────────┬─────────────┐
│ cohort_defin ┆ number_recor ┆ number_subje ┆ reason_id ┆ reason      ┆ excluded_re ┆ excluded_su │
│ ition_id     ┆ ds           ┆ cts          ┆ ---       ┆ ---         ┆ cords       ┆ bjects      │
│ ---          ┆ ---          ┆ ---          ┆ i64       ┆ str         ┆ ---         ┆ ---         │
│ i64          ┆ i64          ┆ i64          ┆           ┆             ┆ i64         ┆ i64         │
╞══════════════╪══════════════╪══════════════╪═══════════╪═════════════╪═════════════╪═════════════╡
│ 1            ┆ 3            ┆ 3            ┆ 1         ┆ Initial     ┆ 0           ┆ 0           │
│              ┆              ┆              ┆           ┆ qualifying  ┆             ┆             │
│              ┆              ┆              ┆           ┆ events      ┆             ┆             │
│ 2            ┆ 6            ┆ 6            ┆ 1         ┆ Initial     ┆ 0   

In [22]:
# Full cohort data — note the cohort_definition_id column
print(multi.collect())

shape: (9, 4)
┌──────────────────────┬────────────┬───────────────────┬─────────────────┐
│ cohort_definition_id ┆ subject_id ┆ cohort_start_date ┆ cohort_end_date │
│ ---                  ┆ ---        ┆ ---               ┆ ---             │
│ i64                  ┆ i64        ┆ date              ┆ date            │
╞══════════════════════╪════════════╪═══════════════════╪═════════════════╡
│ 1                    ┆ 17         ┆ 2023-10-02        ┆ 2023-12-24      │
│ 1                    ┆ 2          ┆ 2023-06-17        ┆ 2023-10-26      │
│ 2                    ┆ 17         ┆ 2000-08-12        ┆ 2023-12-24      │
│ 2                    ┆ 15         ┆ 2005-06-29        ┆ 2024-01-24      │
│ 2                    ┆ 16         ┆ 1989-07-09        ┆ 2024-01-21      │
│ 1                    ┆ 16         ┆ 2023-04-28        ┆ 2024-01-21      │
│ 2                    ┆ 5          ┆ 1991-12-22        ┆ 2023-10-29      │
│ 2                    ┆ 19         ┆ 2003-09-08        ┆ 2024-01-01      

## CIRCE/JSON Cohort Definitions

For more complex cohort logic, `generate_cohort_set` accepts CIRCE JSON
definitions — the same format used by [ATLAS](https://atlas.ohdsi.org) and
the OHDSI ecosystem.

A CIRCE definition specifies:
- **ConceptSets** — the vocabulary concepts to look for
- **PrimaryCriteria** — which domain events qualify (e.g. ConditionOccurrence)
- **ObservationWindow** — required observation time around the event
- **Limits** — PrimaryCriteriaLimit, QualifiedLimit, ExpressionLimit (First/All/Last)
- **InclusionRules** — additional filtering criteria
- **EndStrategy** — how to compute cohort end dates
- **CollapseSettings** — era padding for merging overlapping periods
- **CensorWindow** — optional hard date boundaries

```python
generate_cohort_set(
    cdm,
    cohort_set,   # dict, list of dicts, JSON string, or path
    name="cohort",
)
```

`cohort_set` can be:
- A single CIRCE JSON dict
- A list of such dicts
- A JSON string
- A `Path` to a `.json` file or directory of files

### Building a CIRCE JSON definition

Below we construct a minimal CIRCE cohort for Viral sinusitis. The key
structure mirrors what ATLAS exports.

In [23]:
import json

# Minimal CIRCE JSON for Viral sinusitis (concept 40481087)
sinusitis_circe = {
    "ConceptSets": [
        {
            "id": 0,
            "name": "Viral sinusitis",
            "expression": {
                "items": [
                    {
                        "concept": {
                            "CONCEPT_ID": 40481087,
                            "CONCEPT_NAME": "Viral sinusitis",
                            "CONCEPT_CODE": "",
                            "DOMAIN_ID": "Condition",
                            "VOCABULARY_ID": "SNOMED",
                            "STANDARD_CONCEPT": "S",
                            "INVALID_REASON": "",
                            "CONCEPT_CLASS_ID": "Disorder",
                        },
                        "includeDescendants": False,
                    }
                ]
            },
        }
    ],
    "PrimaryCriteria": {
        "CriteriaList": [{"ConditionOccurrence": {"CodesetId": 0}}],
        "ObservationWindow": {"PriorDays": 0, "PostDays": 0},
        "PrimaryCriteriaLimit": {"Type": "First"},
    },
    "QualifiedLimit": {"Type": "First"},
    "ExpressionLimit": {"Type": "First"},
    "InclusionRules": [],
    "CensoringCriteria": [],
    "CollapseSettings": {"CollapseType": "ERA", "EraPad": 0},
    "CensorWindow": {},
}

print(json.dumps(sinusitis_circe, indent=2)[:500], "...")

{
  "ConceptSets": [
    {
      "id": 0,
      "name": "Viral sinusitis",
      "expression": {
        "items": [
          {
            "concept": {
              "CONCEPT_ID": 40481087,
              "CONCEPT_NAME": "Viral sinusitis",
              "CONCEPT_CODE": "",
              "DOMAIN_ID": "Condition",
              "VOCABULARY_ID": "SNOMED",
              "STANDARD_CONCEPT": "S",
              "INVALID_REASON": "",
              "CONCEPT_CLASS_ID": "Disorder"
            },
           ...


### Generating a CIRCE cohort

In [24]:
# Pass the dict directly — generate_cohort_set parses it
circe_defn = {
    "id": 1,
    "name": "viral_sinusitis_circe",
    "expression": sinusitis_circe,
}

cdm = generate_cohort_set(cdm, circe_defn, name="vs_circe")

vs_circe = cdm["vs_circe"]
print(vs_circe)
print(f"Records: {vs_circe.count()}")

CohortTable('vs_circe', source='duckdb', cohorts=1)
Records: 3


In [25]:
print("Settings:")
print(vs_circe.settings)
print()
print("Attrition:")
print(vs_circe.attrition)

Settings:
shape: (1, 2)
┌──────────────────────┬───────────────────────┐
│ cohort_definition_id ┆ cohort_name           │
│ ---                  ┆ ---                   │
│ i64                  ┆ str                   │
╞══════════════════════╪═══════════════════════╡
│ 1                    ┆ viral_sinusitis_circe │
└──────────────────────┴───────────────────────┘

Attrition:
shape: (6, 7)
┌──────────────┬──────────────┬──────────────┬───────────┬─────────────┬─────────────┬─────────────┐
│ cohort_defin ┆ number_recor ┆ number_subje ┆ reason_id ┆ reason      ┆ excluded_re ┆ excluded_su │
│ ition_id     ┆ ds           ┆ cts          ┆ ---       ┆ ---         ┆ cords       ┆ bjects      │
│ ---          ┆ ---          ┆ ---          ┆ i64       ┆ str         ┆ ---         ┆ ---         │
│ i64          ┆ i64          ┆ i64          ┆           ┆             ┆ i64         ┆ i64         │
╞══════════════╪══════════════╪══════════════╪═══════════╪═════════════╪═════════════╪═════════════╡
│

In [26]:
print("Cohort data:")
print(vs_circe.collect())

Cohort data:
shape: (3, 4)
┌──────────────────────┬────────────┬───────────────────┬─────────────────┐
│ cohort_definition_id ┆ subject_id ┆ cohort_start_date ┆ cohort_end_date │
│ ---                  ┆ ---        ┆ ---               ┆ ---             │
│ i64                  ┆ i64        ┆ date              ┆ date            │
╞══════════════════════╪════════════╪═══════════════════╪═════════════════╡
│ 1                    ┆ 17         ┆ 2023-10-02        ┆ 2023-12-24      │
│ 1                    ┆ 16         ┆ 2023-04-28        ┆ 2024-01-21      │
│ 1                    ┆ 2          ┆ 2023-06-17        ┆ 2023-10-26      │
└──────────────────────┴────────────┴───────────────────┴─────────────────┘


### Passing a JSON string

You can also pass a raw JSON string — useful when loading definitions from files
or APIs.

In [27]:
# Convert to JSON string and pass directly
json_str = json.dumps(sinusitis_circe)

cdm = generate_cohort_set(cdm, json_str, name="vs_from_json")
print(f"From JSON string — records: {cdm['vs_from_json'].count()}")
print(cdm["vs_from_json"].collect())

From JSON string — records: 3
shape: (3, 4)
┌──────────────────────┬────────────┬───────────────────┬─────────────────┐
│ cohort_definition_id ┆ subject_id ┆ cohort_start_date ┆ cohort_end_date │
│ ---                  ┆ ---        ┆ ---               ┆ ---             │
│ i64                  ┆ i64        ┆ date              ┆ date            │
╞══════════════════════╪════════════╪═══════════════════╪═════════════════╡
│ 1                    ┆ 2          ┆ 2023-06-17        ┆ 2023-10-26      │
│ 1                    ┆ 17         ┆ 2023-10-02        ┆ 2023-12-24      │
│ 1                    ┆ 16         ┆ 2023-04-28        ┆ 2024-01-21      │
└──────────────────────┴────────────┴───────────────────┴─────────────────┘


### CIRCE with `limit="All"` — keep all events

In [28]:
# Same concept, but keep all events instead of just the first
sinusitis_all_circe = sinusitis_circe.copy()
sinusitis_all_circe["PrimaryCriteria"] = {
    "CriteriaList": [{"ConditionOccurrence": {"CodesetId": 0}}],
    "ObservationWindow": {"PriorDays": 0, "PostDays": 0},
    "PrimaryCriteriaLimit": {"Type": "All"},
}
sinusitis_all_circe["QualifiedLimit"] = {"Type": "All"}
sinusitis_all_circe["ExpressionLimit"] = {"Type": "All"}

cdm = generate_cohort_set(
    cdm,
    json.dumps(sinusitis_all_circe),
    name="vs_all_circe",
)

print(f"First-only records: {cdm['vs_circe'].count()}")
print(f"All-events records: {cdm['vs_all_circe'].count()}")
print(cdm["vs_all_circe"].collect())

First-only records: 3
All-events records: 3
shape: (3, 4)
┌──────────────────────┬────────────┬───────────────────┬─────────────────┐
│ cohort_definition_id ┆ subject_id ┆ cohort_start_date ┆ cohort_end_date │
│ ---                  ┆ ---        ┆ ---               ┆ ---             │
│ i64                  ┆ i64        ┆ date              ┆ date            │
╞══════════════════════╪════════════╪═══════════════════╪═════════════════╡
│ 1                    ┆ 16         ┆ 2023-04-28        ┆ 2024-01-21      │
│ 1                    ┆ 2          ┆ 2023-06-17        ┆ 2023-10-26      │
│ 1                    ┆ 17         ┆ 2023-10-02        ┆ 2023-12-24      │
└──────────────────────┴────────────┴───────────────────┴─────────────────┘


## Subsetting CDM by Cohort

`cdm_subset_cohort` creates a new CDM containing only persons who appear in a
specified cohort. All clinical tables are filtered to those persons while
remaining lazy (no data is materialised until you call `.collect()`).

```python
cdm_subset_cohort(
    cdm,
    cohort_table="cohort",   # name of the cohort table in the CDM
    cohort_id=None,          # optional: specific cohort_definition_id(s)
)
```

In [29]:
# Subset the CDM to persons in the hypertension cohort
ht_cdm = cdm_subset_cohort(cdm, cohort_table="hypertension")

print(f"Full CDM persons:     {cdm['person'].count()}")
print(f"Subset CDM persons:   {ht_cdm['person'].count()}")

Full CDM persons:     27
Subset CDM persons:   6


In [30]:
# The subset retains all tables, but each is filtered to cohort members
print(f"Condition occurrences (full):   {cdm['condition_occurrence'].count()}")
print(f"Condition occurrences (subset): {ht_cdm['condition_occurrence'].count()}")

Condition occurrences (full):   59
Condition occurrences (subset): 26


In [31]:
# Subset using the multi-cohort table with a specific cohort_id
sinus_cdm = cdm_subset_cohort(
    cdm,
    cohort_table="multi_cohort",
    cohort_id=[1],  # viral_sinusitis is cohort_definition_id 1
)
print(f"Persons in sinusitis cohort: {sinus_cdm['person'].count()}")

Persons in sinusitis cohort: 3


In [32]:
# Inspect the subsetted data
print("Drug exposures for hypertension cohort:")
print(ht_cdm["drug_exposure"].head(5).collect())

Drug exposures for hypertension cohort:
shape: (5, 23)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ drug_expo ┆ person_id ┆ drug_conc ┆ drug_expo ┆ … ┆ drug_sour ┆ drug_sour ┆ route_sou ┆ dose_uni │
│ sure_id   ┆ ---       ┆ ept_id    ┆ sure_star ┆   ┆ ce_value  ┆ ce_concep ┆ rce_value ┆ t_source │
│ ---       ┆ i64       ┆ ---       ┆ t_date    ┆   ┆ ---       ┆ t_id      ┆ ---       ┆ _value   │
│ i64       ┆           ┆ i32       ┆ ---       ┆   ┆ str       ┆ ---       ┆ str       ┆ ---      │
│           ┆           ┆           ┆ date      ┆   ┆           ┆ i32       ┆           ┆ str      │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
│ 13        ┆ 5         ┆ 1127433   ┆ 2023-06-0 ┆ … ┆ 313782    ┆ 1127433   ┆ null      ┆ null     │
│           ┆           ┆           ┆ 6         ┆   ┆           ┆           ┆           ┆          │
│ 14        ┆ 5         ┆ 19078106  

## Summary

This notebook demonstrated the two cohort generation approaches in OMOPy:

| Approach | Function | Use case |
|---|---|---|
| **Concept-based** | `generate_concept_cohort_set` | Quick cohorts from concept IDs — ideal for simple inclusion criteria |
| **CIRCE/JSON** | `generate_cohort_set` | Full ATLAS-style definitions with inclusion rules, end strategies, censoring |

Key parameters for concept-based cohorts:
- `limit` — `"first"` (one entry per person) or `"all"` (every qualifying event)
- `end` — `"observation_period_end_date"`, `"event_end_date"`, or an integer (days)
- `required_observation` — `(prior_days, future_days)` minimum observation time

After generation, `CohortTable` provides `.settings`, `.attrition`,
`.cohort_codelist`, and `.cohort_count()` for inspecting results.

Use `cdm_subset_cohort` to filter the entire CDM to cohort members for
downstream analysis.

**Next steps:** Continue to [04_characterisation.ipynb](./04_characterisation.ipynb)
for patient-level characterisation and incidence analyses on these cohorts.